# Treinar o modelo de deteção de alimentos (Google Colab)

Treina um **YOLOv8s-seg** no dataset FoodSeg103 (103 alimentos) e compara-o com o modelo atual do projeto
(um YOLOv8**n**-seg, ~3x mais pequeno, com mAP50 = 0,215 nas fotos de teste).

**Antes de começar:** `Ambiente de execução` → `Alterar tipo de ambiente de execução` → **GPU T4**.

**Tempo:** ~10 min de preparação + ~1h30 a 2h de treino. Os resultados são guardados no Google Drive,
por isso se o Colab se desligar, não se perde o treino (ver passo 4b).

Conjuntos de dados (ver `scripts/preparar_foodseg103.py`):
- **treino** (90% das fotos de treino) — o modelo aprende com estas
- **validação** (10%) — o YOLO usa-as para escolher a melhor época
- **teste** (2135 fotos oficiais) — só para a comparação final; nunca entram no treino

## 1. Confirmar que há GPU

In [ ]:
!nvidia-smi

## 2. Ligar o Google Drive (para guardar o treino)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PASTA_TREINO = '/content/drive/MyDrive/smart_carb_treino'

## 3. Descarregar o projeto e preparar os dados (~10 min)

In [ ]:
!git clone https://github.com/PituchoX/Smart-Carb-Assistant.git
%cd /content/Smart-Carb-Assistant
!pip install -q ultralytics
!python scripts/preparar_foodseg103.py --treino

## 4. Treinar (~1h30 a 2h)

- `epochs=60`: no máximo 60 passagens pelas fotos de treino
- `patience=15`: para mais cedo se o modelo não melhorar durante 15 épocas
- `yolov8s-seg.pt`: começa de um modelo já treinado em fotos gerais (COCO), o que acelera a aprendizagem

In [ ]:
from ultralytics import YOLO

modelo = YOLO('yolov8s-seg.pt')
modelo.train(
    data='dados/foodseg103/data.yaml',
    epochs=60,
    patience=15,
    imgsz=640,
    batch=16,
    seed=42,
    project=PASTA_TREINO,
    name='yolov8s_foodseg103',
    exist_ok=True,
)

### 4b. Só se o Colab se tiver desligado a meio do treino

Corre outra vez os passos 1 a 3 e depois esta célula: continua a partir da última época guardada no Drive.

In [ ]:
modelo = YOLO(f'{PASTA_TREINO}/yolov8s_foodseg103/weights/last.pt')
modelo.train(resume=True)

## 5. Comparar com o modelo atual nas fotos de teste

Usa o mesmo script e as mesmas 2135 fotos com que o modelo atual teve mAP50 = 0,215.

In [ ]:
# Modelo atual: YOLOv8n-seg de gorkatorres (Hugging Face, Apache 2.0)
!wget -q -O modelo_segmentacao_alimentos.pt https://huggingface.co/gorkatorres/8_Reto2_0ndAAkin_2526_G1/resolve/main/weights/best.pt

MODELO_NOVO = f'{PASTA_TREINO}/yolov8s_foodseg103/weights/best.pt'
!python scripts/avaliar_detecao.py atual "{MODELO_NOVO}"

## 6. Descarregar o modelo novo

Fica também guardado no Google Drive, em `smart_carb_treino/yolov8s_foodseg103/weights/best.pt`.

In [ ]:
from google.colab import files
files.download(MODELO_NOVO)